# Kentrodraught: Rust training handoff

Attach the private starter dataset prepared with this notebook. Weight updates and predictions run in **Rust on Kaggle CPUs**; Python schedules and audits. No GPU/TPU kernels are implemented.

The transferred round-13 corpus and champion already completed four local fits, with no eligible new checkpoint. The default notebook verifies the transfer, builds/tests Rust and checks the corpus. It does **not** rerun those fits as supposed progress. Prepare new deep-teaching targets from the separate research handoff for genuinely new training. Enable Internet only if Rust/Clippy needs installation.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, urllib.request, zipfile

roots = [p.parent for p in Path('/kaggle/input').rglob('run_training.py')
         if (p.parent/'data/corpus.bin').is_file() and (p.parent/'model/eval_nnue.nnk').is_file()]
if not roots:
    archives = list(Path('/kaggle/input').rglob('Kentrodraught-Kaggle-Starter*.zip'))
    if len(archives) != 1:
        raise RuntimeError('Attach the Kentrodraught starter dataset; exactly one starter archive is required.')
    destination = Path('/kaggle/working/kentrodraught-starter'); destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            target = (destination/item.filename).resolve()
            if not target.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Archive contains an unsafe path or symlink')
        archive.extractall(destination)
    roots = [p.parent for p in destination.rglob('run_training.py') if (p.parent/'data/corpus.bin').is_file()]
if len(roots) != 1:
    raise RuntimeError('More than one starter was found; set ROOT explicitly to the intended dataset.')
ROOT = roots[0]
print('Starter:', ROOT)
print('Allowed CPU IDs:', sorted(os.sched_getaffinity(0)))


In [ ]:
# Install the official Rust toolchain only if Kaggle does not have a usable one.
os.environ['PATH'] = str(Path.home()/'.cargo/bin') + os.pathsep + os.environ.get('PATH', '')
if not shutil.which('cargo'):
    installer = Path('/kaggle/working/rustup-init.sh')
    urllib.request.urlretrieve('https://sh.rustup.rs', installer)
    subprocess.run(['sh', str(installer), '-y', '--profile', 'minimal', '--default-toolchain', 'stable'], check=True)
if not shutil.which('cargo'):
    raise RuntimeError('Rust was not installed; enable Kaggle Internet and rerun this cell.')
version = subprocess.check_output(['rustc', '--version'], text=True).split()[1]
if tuple(map(int, version.split('.')[:2])) < (1, 85):
    if not shutil.which('rustup'):
        raise RuntimeError('Rust >=1.85 is required. Install the official stable toolchain.')
    subprocess.run(['rustup', 'toolchain', 'install', 'stable', '--profile', 'minimal', '--component', 'clippy'], check=True)
    os.environ['RUSTUP_TOOLCHAIN'] = 'stable'
if subprocess.run(['cargo', 'clippy', '--version'], capture_output=True).returncode:
    subprocess.run(['rustup', 'component', 'add', 'clippy'], check=True)
print(subprocess.check_output(['rustc', '--version'], text=True))


The next cell verifies model, corpus and sidecar hashes and prints the exact plan. It does not compile or train. Historical absolute paths inside the frozen evidence are provenance; the portable launcher uses files relative to `ROOT`.


In [ ]:
subprocess.run([sys.executable, str(ROOT/'run_training.py'), '--root', str(ROOT),
                '--output', '/kaggle/working/rust-training-001', '--dry-run'], check=True)


Build and test the native Rust trainer, then check the transferred corpus. This runs **no optimizer**. Preserve the preflight outputs.


In [ ]:
OUTPUT = Path('/kaggle/working/rust-preflight-001')
subprocess.run([sys.executable, str(ROOT/'run_training.py'), '--root', str(ROOT),
                '--output', str(OUTPUT), '--check-only'], check=True)


In [ ]:
# Optional reproducibility test only: these exact four fits already finished locally.
RUN_BASELINE_REPRODUCTION = False
if RUN_BASELINE_REPRODUCTION:
    subprocess.run([sys.executable, str(ROOT/'run_training.py'), '--root', str(ROOT),
                    '--output', '/kaggle/working/rust-baseline-reproduction-001',
                    '--workers', '4', '--purpose', 'reproduce'], check=True)


In [ ]:
status = json.loads((OUTPUT/'status.json').read_text())
print(json.dumps({key: status.get(key) for key in ['state', 'completed_trials', 'selected_weights_sha256',
      'selection_reason', 'playing_strength_validated', 'target_reached', 'next_required']}, indent=2))
# Save the notebook with outputs and retain every trial, report and preregistration.
archive_path = shutil.make_archive('/kaggle/working/Kentrodraught-Rust-Training-Results', 'zip', OUTPUT)
print('Download or retain:', archive_path)


Offline improvement is a candidate filter. Before adopting new weights, rebuild the exact Rust engine, verify quantized inference and game rules, pass all 359 certificates, then run preregistered color-balanced 1-second matches against unchanged C++. Enforce the stricter planned defense gate. Seven actual wins in ten fresh confirmation games is the target; cutoffs remain unknown for training and never count as wins. The full research dataset and prepared deep-teaching source are separate from this compact starter.
